# Réduire la redondance des capteurs de chaîne de production avec PROC GVARCLUS

## Synthèse

Une chaîne de fabrication multi-zones diffuse des dizaines de mesures de capteurs corrélées, dont beaucoup portent le même signal sous-jacent. Ce notebook utilise **PROC GVARCLUS** (classification de variables) pour regrouper les 13 capteurs de procédé en quatre grappes disjointes, puis lit les valeurs de R² de chaque grappe pour choisir une jauge représentative par grappe — réduisant l'empreinte de surveillance SPC sans perdre d'information sur le procédé. Trois des quatre grappes correspondent nettement à des zones physiques (R² moyen de 0,92, 0,93 et 0,96) ; la quatrième est une grappe à un seul canal que la procédure a isolée, que nous examinons plutôt que de la passer sous silence.

## Sources de données

Toutes les données sont générées en ligne avec `call streaminit(20260531)` et `rand()` — aucune entrée externe ni réseau.

| Jeu de données | Lignes | Variables clés | Description |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Mesures synthétiques d'une chaîne de production à 3 zones. Les capteurs d'une même zone partagent un facteur latent (forte corrélation) ; des jauges inter-zones (températures liées à la zone 1, pressions à la zone 2, vibration à la zone 3) sont ajoutées pour créer une redondance réaliste. La boucle du DATA step demande 300 cycles, mais cet environnement d'évaluation fonctionne en mode sans licence et ne conserve que les 100 premières observations — suffisant pour retrouver nettement la structure des grappes. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | Une ligne par capteur d'entrée : la grappe à laquelle il a été affecté et son R² avec la composante de cette grappe. Produite par l'instruction `OUTPUT OUT=`. |

# Réduire la redondance des capteurs de chaîne de production avec PROC GVARCLUS

Sur une chaîne de production instrumentée, il est courant d'enregistrer des dizaines de capteurs qui mesurent des phénomènes physiques qui se recoupent — plusieurs thermocouples dans une même zone, des transducteurs de pression redondants, des sondes de vibration qui suivent le même arbre. Faire entrer chaque canal dans une carte de contrôle ou un modèle en aval gaspille l'effort de surveillance et gonfle la multicolinéarité.

La **classification de variables** répond directement à ce problème. `PROC GVARCLUS` partitionne les variables numériques en grappes *disjointes*, où chaque grappe est résumée par la première composante principale de ses membres. Les capteurs qui évoluent ensemble atterrissent dans la même grappe ; l'ingénieur peut alors ne garder qu'un canal représentatif par grappe.

Dans ce notebook, nous :

1. Synthétisons des mesures d'une chaîne à 3 zones avec des capteurs délibérément redondants (la boucle demande 300 cycles ; 100 sont conservés ici).
2. Regroupons les 13 canaux avec `PROC GVARCLUS` à `MAXCLUSTERS=4`.
3. Capturons les affectations de grappe dans un jeu de données de sortie et les résumons.
4. Interprétons les valeurs de R² pour choisir une jauge par grappe pour la surveillance SPC continue.

## Étape 1 — Générer des données de capteurs synthétiques

Nous simulons trois zones de procédé, chacune avec un facteur caché (`zone1_a`, `zone2_a`, `zone3_a`). Les autres canaux sont des fonctions linéaires bruitées du facteur de leur zone, si bien qu'ils sont fortement corrélés au sein d'une zone mais quasi indépendants d'une zone à l'autre. Nous lions aussi les températures d'entrée/sortie à la zone 1 et les deux transducteurs de pression à la zone 2, imitant la redondance inter-instruments observée sur des chaînes réelles. Une graine fixe rend l'exécution reproductible. La boucle demande 300 cycles ; en mode sans licence, le moteur conserve les 100 premières observations, ce que rapporte la NOTE ci-dessous.

In [1]:
DONNÉES process_sensors;
    APPELER streaminit(20260531);
    FAIRE cycle = 1 JUSQU_À 300;
        /* Zone 1: latent driver plus two redundant probes */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zone 2: latent driver plus two redundant probes */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zone 3: latent driver plus one redundant probe */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Cross-instrument channels tied to existing drivers */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        SORTIE;
    FIN;
    SUPPRIMER cycle;
EXÉCUTER;


NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds


## Étape 2 — Regrouper les capteurs

Nous appelons `PROC GVARCLUS` sur les 13 canaux. L'instruction `VAR` liste les capteurs numériques à regrouper. Nous plafonnons la partition à quatre grappes avec `MAXCLUSTERS=4` (nous nous attendons à environ une grappe par zone physique, avec un peu de marge). `ODS GRAPHICS ON` avec `PLOTS=ALL` active le dendrogramme de grappes de variables ; le moteur écrit ce SVG dans le répertoire de travail plutôt que de le restituer en ligne, si bien que la structure que nous lisons ci-dessous provient du tableau imprimé **Variable Cluster Assignments** et du tableau des valeurs propres par grappe.

L'instruction `OUTPUT OUT=` écrit les affectations variable-à-grappe — avec le R² de chaque variable par rapport à la composante de sa propre grappe — dans un jeu de données que l'on pourra exploiter par la suite.

In [2]:
ODS GRAPHICS SUR;

PROCÉDURE gvarclus DONNÉES=process_sensors maxclusters=4 PLOTS=TOUT;
    VAR zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    SORTIE out=clusters;
EXÉCUTER;

ODS GRAPHICS OFF;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Étape 3 — Relancer avec l'option SUMMARY

Relancer la procédure une seconde fois avec l'option `SUMMARY` conserve la même partition. `PLOTS=NONE` désactive les graphiques pour ce passage. Dans cet environnement, le rapport imprimé est identique à celui de l'étape 2 — le même tableau d'affectation à 13 lignes, les mêmes quatre grappes et le même résumé des valeurs propres — si bien que cette cellule démontre surtout que les options `SUMMARY` et `PLOTS=NONE` s'analysent et s'exécutent correctement ; elle n'est pas censée ajouter de nouveaux chiffres.

In [3]:
PROCÉDURE gvarclus DONNÉES=process_sensors maxclusters=4 summary PLOTS=none;
    VAR zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
EXÉCUTER;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Étape 4 — Examiner le jeu de données de sortie

Le jeu de données `clusters` issu de `OUTPUT OUT=` comporte une ligne par capteur avec sa `Cluster` affectée et son `RSq_Own` (la corrélation au carré entre le capteur et la composante de sa grappe). Un `RSq_Own` élevé signifie que le capteur est bien représenté par sa grappe — un candidat idéal à *abandonner* au profit du représentant de la grappe. Nous trions par grappe puis par R², afin que le capteur le plus représentatif de chaque grappe se trouve en tête de son groupe.

In [4]:
PROCÉDURE TRIER DONNÉES=clusters out=clusters_ranked;
    PAR Cluster DESCENDANT RSq_Own;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=clusters_ranked ÉTIQUETTE;
    VAR Variable Cluster RSq_Own;
    ÉTIQUETTE Variable = "Canal du capteur"
          Cluster  = "Grappe"
          RSq_Own  = "R² avec sa propre grappe";
EXÉCUTER;


  Obs  Canal du capteur  Grappe   R² avec sa propre grappe
-----  ----------------  ------  -------------------------
    1  ZONE1_A                1                    0.96867
    2  ZONE1_B                1                     0.9189
    3  TEMP_IN                1                   0.903394
    4  TEMP_OUT               1                   0.889519
    5  ZONE2_A                2                    0.98364
    6  ZONE2_B                2                   0.946708
    7  PRESSURE_A             2                   0.929356
    8  PRESSURE_B             2                   0.920915
    9  ZONE2_C                2                   0.892405
   10  ZONE3_A                3                   0.977237
   11  VIBRATION              3                    0.95916
   12  ZONE3_B                3                   0.949054
   13  ZONE1_C                4                          1




NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## Interpréter les résultats

La partition retrouve l'essentiel de la structure physique de la chaîne, avec une réserve honnête à signaler :

- La **grappe 1** rassemble `zone1_a` (R²=0,969), `zone1_b` (0,919) et les températures d'entrée/sortie `temp_in` (0,903) et `temp_out` (0,890) — quatre des cinq canaux pilotés par le signal latent de la zone 1. R² moyen 0,920.
- La **grappe 2** rassemble les trois canaux de la zone 2 `zone2_a` (0,984), `zone2_b` (0,947), `zone2_c` (0,892) ainsi que les deux transducteurs de pression `pressure_a` (0,929) et `pressure_b` (0,921), que nous avons liés au facteur de la zone 2. R² moyen 0,935.
- La **grappe 3** rassemble `zone3_a` (0,977), `zone3_b` (0,949) et `vibration` (0,959). R² moyen 0,962.
- La **grappe 4** est un canal unique : `zone1_c`, la troisième sonde de la zone 1, a été isolée seule avec R²=1,000 (un singleton est, trivialement, parfaitement expliqué par lui-même). Bien qu'il partage le facteur de la zone 1, à cette taille d'échantillon la procédure a jugé `zone1_c` suffisamment distinct du groupe `zone1_a`/températures pour justifier sa propre grappe plutôt que de le fusionner dans la grappe 1.

Les trois grappes multi-canaux affichent chacune un R² moyen supérieur à **0,90**, confirmant qu'une seule composante explique l'essentiel de la variation entre leurs membres — exactement la redondance que l'on veut réduire. Le seul cas atypique — `zone1_c` isolé dans sa propre grappe plutôt qu'avec le reste de la zone 1 — rappelle utilement que la classification de variables est pilotée par les données : avec plus de cycles ou un budget `MAXCLUSTERS` plus élevé, la frontière peut se déplacer, si bien que la partition est un point de départ pour le jugement d'ingénierie, pas une vérité figée.

**Action pour la chaîne.** Dans chaque grappe multi-canaux, garder le capteur avec le `RSq_Own` le plus élevé (le canal le plus représentatif de sa grappe) et retirer ou déprioriser les autres du suivi SPC de routine — par exemple `zone2_a` pour la grappe 2 et `zone3_a` pour la grappe 3. Traiter le singleton `zone1_c` comme un signal à investiguer plutôt qu'un maintien automatique : confirmer s'il porte une information réellement distincte ou s'il s'agit d'un artefact de la classification avant de décider de le surveiller séparément. Même en mettant ce canal de côté, cela réduit 13 canaux surveillés vers un plan de surveillance à quatre canaux, diminuant le bruit d'alarme et la multicolinéarité tout en préservant le contenu informationnel de la chaîne.